# linear-affine-on-custom-tensor — ex2: bias backward for Linear: unbroadcast grad over batch axis

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `linear-affine-on-custom-tensor`. Running the final beacon cell reports progress against the `Backprop: Linear affine on custom Tensor` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Linear affine on custom Tensor` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`linear-affine-on-custom-tensor`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "linear-affine-on-custom-tensor"
DD_SUBTOPIC = "Backprop: Linear affine on custom Tensor"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Linear-affine backward — bias path — deepening

Forward: `out = x @ W + b` broadcasts `b: (out_f,)` over the batch axis to produce `out: (B, out_f)`. Backward through that broadcast is the **inverse** operation: an upstream gradient `grad_out` of shape `(B, out_f)` must collapse back to `(out_f,)` for `b`'s grad.

The collapse rule is: **sum over every axis that got broadcast.** Here that's the leading batch axis:

```python
grad_b = grad_out.sum(dim=0)   # (B, out_f) -> (out_f,)
```

If `b` had been `(1, out_f)` instead (keepdim broadcast), the rule would be `grad_out.sum(dim=0, keepdim=True)` — preserve the same shape the forward broadcast started from.

### Exercise 2 — bias backward for Linear: unbroadcast grad over batch axis

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the unbroadcast-via-sum rule to collapse an upstream (B, out_f) gradient back to (out_f,) — the bias-backward for an affine Linear layer over MiniTensors.
> Keywords: linear, bias-backward, unbroadcast, sum-over-batch
> ```

**KCs targeted:** `linear-affine-on-custom-tensor`, `unbroadcast-via-sum`

Implement `linear_bias_backward(grad_out, bias)`. This is the **reverse-pass contribution to the bias** in the affine map `out = mm + bias`.

Inputs:
- `grad_out`: a raw `torch.Tensor` of shape `(B, out_f)` — the upstream gradient flowing into `out`.
- `bias`:     a `MiniTensor` of shape `(out_f,)` — the parameter we're computing grad for (passed only for shape reference).

Behavior:
1. Sum `grad_out` over the batch axis (axis 0). The bias broadcast over the batch, so the backward collapses over the batch.
2. Confirm the result shape equals `bias.array.shape` — if it doesn't, your unbroadcast is wrong.
3. Return the raw `torch.Tensor` of shape `(out_f,)`.

Return type is a plain tensor, NOT a MiniTensor — backward contributions are raw tensors that the reverse-pass dispatcher accumulates into a `grads` dict.

**Why this is the deepening of ex1.** Ex1 built the forward `mm + bias` Recipe chain; ex2 builds the backward primitive that the dispatcher will pair with that Recipe. Together they make a fully end-to-end-trainable Linear.

In [ ]:
def linear_bias_backward(grad_out, bias: MiniTensor):
    """Collapse a (B, out_f) grad to (out_f,) by summing the batch axis."""
    raise NotImplementedError()


def _test_ex2():
    # --- invariant 1: shape collapse (B, out_f) -> (out_f,) ---
    B, out_f = 4, 5
    bias = MiniTensor(t.zeros(out_f), requires_grad=True)
    grad_out = t.ones(B, out_f)
    gb = linear_bias_backward(grad_out, bias)
    assert isinstance(gb, t.Tensor), f'must return raw torch.Tensor, got {type(gb).__name__}'
    assert gb.shape == (out_f,), f'shape: {gb.shape}'

    # --- invariant 2: ones-input collapses to all-B value (sum, not mean) ---
    assert t.allclose(gb, t.full((out_f,), float(B))), (
        f'sum-over-batch of all-ones must equal B for each out-channel, got {gb}'
    )

    # --- invariant 3: matches torch.autograd reference on a real affine forward ---
    x_ref = t.randn(B, 3)
    w_ref = t.randn(3, out_f, requires_grad=True)
    b_ref = t.randn(out_f, requires_grad=True)
    out_ref = x_ref @ w_ref + b_ref
    grad_seed = t.randn(B, out_f)
    out_ref.backward(grad_seed)
    ours = linear_bias_backward(grad_seed, MiniTensor(b_ref.detach()))
    assert t.allclose(ours, b_ref.grad, atol=1e-5), (
        f'must match torch autograd on the same seed; ours={ours} vs ref={b_ref.grad}'
    )

    # --- invariant 4: non-trivial batch size, dtype preserved ---
    g64 = t.randn(32, 10, dtype=t.float64)
    b64 = MiniTensor(t.zeros(10, dtype=t.float64))
    gb64 = linear_bias_backward(g64, b64)
    assert gb64.shape == (10,) and gb64.dtype == t.float64, f'dtype/shape: {gb64.dtype}/{gb64.shape}'
    assert t.allclose(gb64, g64.sum(dim=0))
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def linear_bias_backward(grad_out, bias: MiniTensor):
    # bias broadcast over axis 0 in the forward -> sum over axis 0 in reverse
    gb = grad_out.sum(dim=0)
    assert gb.shape == bias.array.shape, (
        f'bias-grad shape {gb.shape} must equal bias shape {bias.array.shape}'
    )
    return gb
```

**Unbroadcast = sum over expanded axes.** The general rule: if the forward used `a + b` where `b` was implicitly expanded along axes `E`, the backward to `b` is `grad_out.sum(dim=E)`. For Linear's bias the expansion is the batch axis, so `sum(dim=0)` recovers the right shape.

**Why the shape assert.** Catching a shape mismatch at the backward boundary is the cheapest debugging — most autograd bugs show up as 'shape (B, out_f) cannot be assigned to bias.grad of shape (out_f,)' three frames deeper. Failing fast here saves 30 seconds of stack-walking.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()